# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [pareshmishra23](https://github.com/pareshmishra23)

Github repository: https://github.com/pareshmishra23/4k-genration

Original by: [yuvraj108c](https://github.com/yuvraj108c)

---
**Note:** Make sure to change Runtime to GPU — Runtime → Change runtime type → T4 GPU


# 1. Setup (~1 minute)

In [ ]:
import os, sys, pathlib, glob, subprocess
import torch

# Check GPU availability
if not torch.cuda.is_available():
    print("⚠️ WARNING: GPU not detected. Please change runtime to GPU for best performance.")
    print("   Go to: Runtime → Change runtime type → GPU (T4)")
else:
    print("✅ GPU detected:", torch.cuda.get_device_name(0))

from PIL import Image
import cv2
from tqdm import tqdm

# Clone Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Fix numpy version first (critical for basicsr compatibility)
!pip uninstall -y numpy
!pip install -q numpy==1.26.4

# Install torch with GPU support
!pip install -q torch==2.2.2+cu118 torchvision==0.17.2+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

# Install dependencies
!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

# Patch deprecated torchvision import in basicsr
basicsr_path = glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py')
if basicsr_path:
    degradations = pathlib.Path(basicsr_path[0])
    content = degradations.read_text()
    if 'functional_tensor' in content:
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        degradations.write_text(content)
        print('✅ Patched basicsr compatibility')

# Verify everything works
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
print('✅ All dependencies installed successfully!')
print('✅ GPU available:', torch.cuda.is_available())

# Go back to content
%cd /content

mount_drive = False


# 2. Mount drive (optional)

Enable this if you want to read videos from and save to Google Drive.


In [ ]:
from google.colab import drive
mount_drive=True #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

# 3. Upscale video

Configure your settings below and click **Play** to run.

- **video_path**: Path to your input video (supports Google Drive paths)
- **output_dir**: Directory to save the upscaled video
- **resolution**: Target output resolution
- **model**: Choose the upscaling model

If Google Drive is mounted, the output will also be saved to `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [ ]:
# ===================== CONFIGURATION =====================
video_path="/content/gdrive/MyDrive/content/video.mp4" #@param{type:"string"}
output_dir="/content/gdrive/MyDrive/content/" #@param{type:"string"}
resolution = "4k (3840 x 2160)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus_anime_6B" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]
# =========================================================

# Verify video exists
assert os.path.exists(video_path), f"Video file does not exist: {video_path}"
print(f"✅ Video found: {video_path}")

# Get video info
video_capture = cv2.VideoCapture(video_path)
video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video_capture.get(cv2.CAP_PROP_FPS)
total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
video_capture.release()
print(f"✅ Input: {video_width}x{video_height} | {fps} FPS | {duration:.1f}s | {total_frames} frames")

# Calculate target resolution
final_width = None
final_height = None
aspect_ratio = float(video_width / video_height)

# Get output resolutions
match resolution:
  case "FHD (1920 x 1080)":
    final_width=1920
    final_height=1080
  case "2k (2560 x 1440)":
    final_width=2560
    final_height=1440
  case "4k (3840 x 2160)":
    final_width=3840
    final_height=2160
  case "2 x original":
    final_width=2*video_width
    final_height=2*video_height
  case "3 x original":
    final_width=3*video_width
    final_height=3*video_height
  case "4 x original":
    final_width=4*video_width
    final_height=4*video_height

if aspect_ratio == 1.0 and "original" not in resolution:
  final_height = final_width

if aspect_ratio < 1.0 and "original" not in resolution:
  temp = final_width
  final_width = final_height
  final_height = temp

scale_factor = max(final_width/video_width, final_height/video_height)
isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

# scale_factor needs to be even
while isEven == False:
  scale_factor += 0.01
  isEven = int(video_width * scale_factor) % 2 == 0 and int(video_height * scale_factor) % 2 == 0

print(f"Upscaling from {video_width}x{video_height} to {final_width}x{final_height}, scale_factor={scale_factor}")

# ---- Model setup ----
if model == "RealESRGAN_x4plus":
    from basicsr.archs.rrdbnet_arch import RRDBNet
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    netscale = 4
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
elif model == "RealESRGAN_x4plus_anime_6B":
    from basicsr.archs.rrdbnet_arch import RRDBNet
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
    netscale = 4
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth"
elif model == "realesr-animevideov3":
    from realesrgan.archs.srvgg_arch import SRVGGNetCompact
    net = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type="prelu")
    netscale = 4
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth"

# Download model weights
from basicsr.utils.download_util import load_file_from_url
model_path = load_file_from_url(url=file_url, model_dir='/content/Real-ESRGAN/weights', progress=True, file_name=None)
print(f"✅ Model downloaded: {model_path}")

# Create upsampler
from realesrgan import RealESRGANer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
upsampler = RealESRGANer(
    scale=netscale,
    model_path=model_path,
    dni_weight=None,
    model=net,
    tile=128,          # tile size to avoid OOM on GPU
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),  # half precision on GPU, fp32 on CPU
    device=device,
)
print(f"✅ Upsampler ready on {device}")

# ---- Process video frame by frame ----
os.makedirs(output_dir, exist_ok=True)
video_name = os.path.splitext(os.path.basename(video_path))[0]
out_width = int(video_width * scale_factor)
out_height = int(video_height * scale_factor)

video_capture = cv2.VideoCapture(video_path)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
temp_path = os.path.join(output_dir, f'{video_name}_out.mp4')
out_writer = cv2.VideoWriter(temp_path, fourcc, fps, (out_width, out_height))

pbar = tqdm(total=total_frames, desc='Upscaling')
frame_idx = 0
while True:
    ret, frame = video_capture.read()
    if not ret:
        break
    try:
        output, _ = upsampler.enhance(frame, outscale=scale_factor)
        out_writer.write(output)
    except RuntimeError as e:
        print(f'Error on frame {frame_idx}: {e}')
        print('Try reducing tile size or switching model.')
        # Fallback: write resized frame
        resized = cv2.resize(frame, (out_width, out_height))
        out_writer.write(resized)
    pbar.update(1)
    frame_idx += 1

video_capture.release()
out_writer.release()
pbar.close()

# ---- Crop to final resolution ----
final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}.mp4"
final_video_path = os.path.join(output_dir, final_video_name)

if "original" not in resolution:
  print("Cropping to fit...")
  command = f"ffmpeg -loglevel error -y -i '{temp_path}' -filter:v  'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
  subprocess.run(command, shell=True)
else:
  command = f"cp '{temp_path}' '{final_video_path}'"
  subprocess.run(command, shell=True)

# Cleanup temp file
if os.path.exists(temp_path):
    os.remove(temp_path)

print(f"Upscaled video saved to: {final_video_path}")

# ---- Save to Google Drive ----
if mount_drive:
  drive_folder = "MyDrive/Upscaled Videos (REAL-ESRGAN)"
  save_directory_drive = f"/content/gdrive/{drive_folder}"
  os.makedirs(save_directory_drive, exist_ok=True)

  command = f"cp '{final_video_path}' '{save_directory_drive}/{final_video_name}'"
  subprocess.run(command, shell=True)
  print(f"Saved to drive: /{drive_folder}/{final_video_name}")

print("\n🎉 DONE! Your upscaled video is ready.")


# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()